# Protokol evaluasi warm-start

Protokol ini menentukan siapa yang diuji (evaluable users) dan item apa yang boleh direkomendasikan (kandidat warm). Semua model memakai protokol yang sama supaya perbandingannya adil. Hanya train dan validation yang dipakai; test tidak digunakan.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.protocol import build_protocol, protocol_counts, compare_with_official, save_log

D:\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Muat data dan bangun protokol

`build_protocol` membaca train, validation, dan kandidat warm, lalu menyusun dua hal: `seen` (item yang sudah dilihat tiap user di train) dan `relevant` (item relevan tiap user di validation: rating >= 4 dan item termasuk kandidat warm). Evaluable users mengikuti aturan resmi paket proyek: user punya minimal satu item disukai (rating >= 4) di train dan minimal satu di validation. Kunci `relevant` adalah daftar evaluable users.

In [2]:
train, val, cand_set, seen, relevant = build_protocol(str(ROOT / 'data') + '/')
pos_train = set(train.loc[train['rating'] >= 4, 'user_id'])
assert all(u in pos_train for u in relevant), 'ada evaluable user tanpa item disukai di train'
print(f'train={len(train):,}; validation={len(val):,}; kandidat warm={len(cand_set):,}; evaluable users={len(relevant):,}')

train=427,957; validation=41,284; kandidat warm=22,753; evaluable users=11,659


## Ringkasan angka protokol

In [3]:
counts = protocol_counts(train, val, cand_set, seen, relevant)
pd.Series(counts, name='nilai').to_frame()

,nilai
train_interactions,427957
validation_interactions,41284
train_users,54483
train_items,22753
candidate_items_warm,22753
candidates_not_seen_in_train,0
evaluable_users,11659
relevant_pairs,24202
min_rating_relevant,4
top_k,10


## Contoh satu evaluable user

Memperlihatkan bagaimana aturan bekerja untuk satu user: item yang sudah dilihat di train dibuang dari kandidat, dan item relevan di validation harus masih ada di kandidat yang tersisa.

In [4]:
user = next(iter(relevant))
pool = cand_set - seen[user]
print('user:', user)
print('item yang sudah dilihat di train :', len(seen[user]))
print('item relevan di validation       :', sorted(relevant[user]))
print('kandidat yang boleh direkomendasi:', len(pool), 'dari', len(cand_set))
print('item relevan ada di kandidat user:', relevant[user] <= pool)

user: AE22B7BMXT7GAGY3XLHICMESNAPA
item yang sudah dilihat di train : 7
item relevan di validation       : ['B0BYNM15YB']
kandidat yang boleh direkomendasi: 22746 dari 22753
item relevan ada di kandidat user: True


## Cek aturan untuk semua evaluable users

Memastikan setiap evaluable user memenuhi aturan protokol. Kalau ada satu saja yang melanggar, sel ini berhenti dengan error.

In [5]:
assert all(u in seen for u in relevant), 'ada evaluable user yang tidak ada di train'
assert all(len(items) >= 1 for items in relevant.values()), 'ada user tanpa item relevan'
assert all(items <= cand_set for items in relevant.values()), 'ada item relevan di luar kandidat'
assert all(not (items & seen[u]) for u, items in relevant.items()), 'ada item relevan yang sudah dilihat di train'
print('Semua', len(relevant), 'evaluable users memenuhi aturan.')

Semua 11659 evaluable users memenuhi aturan.


## Bandingkan dengan file resmi dosen

Daftar evaluable users kita dicocokkan dengan `evaluable_users_validation_warm.csv` dari paket proyek (folder SUPPORTING_DATA). Hasilnya harus `identical: True`.

In [6]:
official_val = pd.read_csv(ROOT / 'data' / 'interactions_validation_warm.csv')
ours = val[val['user_id'].isin(seen) & val['item_id'].isin(cand_set)]
off_pairs = set(zip(official_val['user_id'], official_val['item_id']))
our_pairs = set(zip(ours['user_id'], ours['item_id']))
print('baris resmi:', len(official_val), '| baris kita:', len(ours))
print('pasangan sama persis:', off_pairs == our_pairs)
print('hanya di resmi:', len(off_pairs - our_pairs), '| hanya di kita:', len(our_pairs - off_pairs))
print('daftar evaluable users:', compare_with_official(relevant, str(ROOT / 'data') + '/'))

baris resmi: 30360 | baris kita: 30360
pasangan sama persis: True
hanya di resmi: 0 | hanya di kita: 0
daftar evaluable users: {'official_file_found': True, 'official_count': 11659, 'our_count': 11659, 'identical': True, 'only_in_official': 0, 'only_in_ours': 0}


## Simpan log

Angka protokol disimpan ke `outputs/logs/protocol_counts.json` sebagai bukti untuk laporan.

In [7]:
save_log(counts, str(ROOT / 'outputs' / 'logs' / 'protocol_counts.json'))

'D:\\Kuliah\\Semester 7\\SISREK\\Proyek\\recsys-lexical-vs-semantic-g11\\outputs\\logs\\protocol_counts.json'